# 03 - Threshold Optimization and Final Evaluation

**Purpose:** Optimize the decision threshold for the tuned Random Forest, lock it in, and then perform the one and only final evaluation on the untouched test set.

**Sections:**
1. Setup & Data Load
2. Preprocessing
3. Retrain the best tuned model
4. Threshold sweep and selection
5. Lock threshold and final test evaluation

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

from src.preprocessing import build_preprocessor
from src.train import build_random_forest
from src.evaluate import evaluate_model, print_metrics
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
sns.set_style("whitegrid")

DATA_DIR = Path("..") / "data" / "raw"
RANDOM_STATE = 42
TEST_SIZE = 0.2

# --- 1. Load and prep (Exactly the same as notebook 02) ---
train = pd.read_csv(DATA_DIR / "fraudTrain.csv")
test = pd.read_csv(DATA_DIR / "fraudTest.csv")
df = pd.concat([train, test], ignore_index=True)
del train, test

df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df["hour"] = df["trans_date_trans_time"].dt.hour
df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek
df["month"] = df["trans_date_trans_time"].dt.month

DROP_COLS = [
    "Unnamed: 0", "cc_num", "first", "last", "street", "city", "zip", "dob", "trans_num",
    "merchant", "job", "unix_time", "trans_date_trans_time", "lat", "long", "merch_lat", "merch_long"
]
TARGET = "is_fraud"

df_model = df.drop(columns=DROP_COLS).copy()
X = df_model.drop(columns=TARGET)
y = df_model[TARGET]

# --- 2. Split (Exactly the same as notebook 02) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE
)

# --- 3. Preprocess ---
pre_tree = build_preprocessor("tree")
X_train_tree = pre_tree.fit_transform(X_train)
X_test_tree  = pre_tree.transform(X_test)

X_tr_tree, X_val_tree, _, _ = train_test_split(
    X_train_tree, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE
)

print("Data ready.")
print(f"Train: {X_tr_tree.shape}, Val: {X_val_tree.shape}, Test: {X_test_tree.shape}")

Data ready.
Train: (1111436, 72), Val: (370479, 72), Test: (370479, 72)


## 3. Retrain the tuned model

In Step 10, `RandomizedSearchCV` found the best parameters:
`{'max_depth': 20, 'max_features': 0.5, 'min_samples_leaf': 15, 'n_estimators': 50}`.

We'll instantiate a fresh Random Forest with exactly those settings and fit it on the training split (`X_tr_tree`, `y_tr`). 

Since this is the final model we're carrying forward, it needs to be trained on the training data, and then we'll use the **validation set** to figure out the best threshold. The test set remains completely untouched.

In [3]:
from sklearn.ensemble import RandomForestClassifier

# The winning parameters from Step 10
best_params = {
    'max_depth': 20,
    'max_features': 0.5,
    'min_samples_leaf': 15,
    'n_estimators': 50
}

# Build and fit the final tuned model
print("Training final tuned Random Forest...")
rf_final = RandomForestClassifier(
    **best_params,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf_final.fit(X_tr_tree, y_tr)

# Get probabilities on the validation set (we need these for the threshold sweep)
val_proba = rf_final.predict_proba(X_val_tree)[:, 1]

print("Training complete.")
print(f"Validation probabilities shape: {val_proba.shape}")
print(f"Min prob: {val_proba.min():.5f} | Max prob: {val_proba.max():.5f}")

Training final tuned Random Forest...
Training complete.
Validation probabilities shape: (370479,)
Min prob: 0.00000 | Max prob: 1.00000
